In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import IsolationForest
import os
import argparse
import sys
import logging

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [3]:
def load_data(input_path: str) -> pd.DataFrame:
    logger.info(f"Memuat dataset dari: {input_path}")
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"File tidak ditemukan: {input_path}")
    df = pd.read_csv(input_path)
    logger.info(f"Dataset berhasil dimuat. Shape: {df.shape}")
    return df

In [4]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Menangani missing values dengan median (numerik) dan modus (kategorikal)."""
    logger.info("Menangani missing values...")
    df = df.copy()

    numerical_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
    for col in numerical_cols:
        if col in df.columns and df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            logger.info(f"  Kolom '{col}' diisi median: {median_val:.4f}")

        if 'label' in df.columns and df['label'].isnull().sum() > 0: modus_val = df['label'].mode()[0]
        df['label'].fillna(modus_val, inplace=True)
        logger.info(f"  Kolom 'label' diisi modus: {modus_val}")

    logger.info(f"Missing values tersisa: {df.isnull().sum().sum()}")
    return df

In [5]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Menghapus baris duplikat."""
    logger.info("Menghapus duplikat...")
    before = df.shape[0]
    df = df.drop_duplicates().reset_index(drop=True)
    after = df.shape[0]
    logger.info(f"  Duplikat dihapus: {before - after} baris")
    return df

In [6]:
def remove_outliers(df: pd.DataFrame, contamination: float = 0.02) -> pd.DataFrame:
    """Mendeteksi dan menghapus outlier menggunakan Isolation Forest."""
    logger.info("Mendeteksi dan menghapus outlier (Isolation Forest)...")
    numerical_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

    before = df.shape[0]
    iso_forest = IsolationForest(contamination=contamination, random_state=42)
    outlier_labels = iso_forest.fit_predict(df[numerical_cols])

    df = df[outlier_labels == 1].copy().reset_index(drop=True)
    after = df.shape[0]
    logger.info(f"  Outlier dihapus: {before - after} baris")
    return df

In [7]:
def encode_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Melakukan label encoding pada kolom target."""
    logger.info("Melakukan label encoding...")
    df = df.copy()
    le = LabelEncoder()
    df['label_encoded'] = le.fit_transform(df['label'])
    logger.info(f"  Jumlah kelas: {len(le.classes_)}")
    return df

In [8]:
def normalize_features(df: pd.DataFrame) -> pd.DataFrame:
    """Normalisasi fitur numerik menggunakan MinMaxScaler."""
    logger.info("Melakukan normalisasi MinMaxScaler...")
    numerical_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
    df = df.copy()
    scaler = MinMaxScaler()
    df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
    logger.info("  Normalisasi selesai. Rentang nilai sekarang [0, 1]")
    return df


In [9]:
def save_data(df: pd.DataFrame, output_path: str) -> None:
    """Menyimpan dataset yang sudah diproses."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    logger.info(f"Dataset disimpan di: {output_path}")
    logger.info(f"Shape akhir: {df.shape}")

In [10]:
def run_preprocessing(input_path: str, output_path: str) -> pd.DataFrame:
    """
    Fungsi utama yang menjalankan seluruh pipeline preprocessing.

    Args:
        input_path  : path ke file CSV raw
        output_path : path untuk menyimpan hasil preprocessing

    Returns:
        DataFrame hasil preprocessing
    """
    logger.info("=" * 50)
    logger.info("MEMULAI PIPELINE PREPROCESSING")
    logger.info("=" * 50)

    df = load_data(input_path)
    df = handle_missing_values(df)
    df = remove_duplicates(df)
    df = remove_outliers(df)
    df = encode_labels(df)
    df = normalize_features(df)
    save_data(df, output_path)

    logger.info("=" * 50)
    logger.info("PREPROCESSING SELESAI")
    logger.info("=" * 50)
    return df

In [11]:
parser = argparse.ArgumentParser()
parser.add_argument("--input", default="../crop_recommendation.csv")
parser.add_argument("--output", default="crop_recommendation_preprocessing.csv")

args, _ = parser.parse_known_args()

INPUT_PATH = args.input
OUTPUT_PATH = args.output